# Automatic Speech Recognition — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def hz_to_mel(f): return 2595*np.log10(1+np.asarray(f)/700)
def mel_to_hz(m): return 700*(10**(np.asarray(m)/2595)-1)
def stft_mag(x,N=256,H=64):
    w=np.hanning(N); frames=[]
    for s in range(0,len(x)-N+1,H): frames.append(np.abs(np.fft.rfft(x[s:s+N]*w)))
    return np.array(frames).T
print('setup ok')

## Acoustic evidence plus temporal decoding
ASR turns frame probabilities into transcript decisions; these examples isolate posterior scores, forward decoding, edit distance, language priors, and a tiny end-to-end utterance.

In [ ]:
# 1) frame logits become posterior evidence and cross-entropy
logits=np.array([1.2,0.3,-0.7]); p=softmax(logits); ce=-math.log(p[0])
plt.figure(figsize=(6,3)); plt.bar(['a','e','s'],p); plt.ylim(0,1); plt.title(f'correct-symbol CE={ce:.3f}')
assert np.allclose(np.round(p,3),[0.643,0.261,0.096]) and abs(ce-0.442)<0.001

In [ ]:
# 2) HMM-style forward recursion combines emissions with duration
T=np.array([[0.7,0.2,0.1],[0.2,0.6,0.2],[0.1,0.2,0.7]]); emis=np.array([[0.8,0.1,0.1],[0.2,0.7,0.1],[0.1,0.2,0.7]])
alpha=np.array([1.0,0,0]); hist=[]
for e in emis:
    alpha=(alpha@T)*e; alpha=alpha/alpha.sum(); hist.append(alpha.copy())
plt.figure(figsize=(6,3)); plt.plot(np.array(hist),'-o'); plt.legend(['state0','state1','state2']); plt.xlabel('frame'); plt.ylabel('posterior'); plt.title('state belief moves through time')
assert np.allclose(np.round(alpha,3),[0.173,0.329,0.497])

In [ ]:
# 3) word error rate counts transcript edits, not frame errors
ref='THE CAT SAT THERE'.split(); hyp='THE BAT SAT DOWN THERE'.split(); S,D,I,N=1,0,1,len(ref); wer=(S+D+I)/N
plt.figure(figsize=(6,3)); plt.bar(['S','D','I'],[S,D,I]); plt.ylim(0,2); plt.title(f'WER={(S+D+I)}/{N}={wer:.3f}')
assert abs(wer-0.5)<1e-9

In [ ]:
# 4) a language prior can change the decoded word
acoustic=np.array([0.55,0.45]); lm=np.array([0.5,0.9]); combined=acoustic*lm; combined=combined/combined.sum()
plt.figure(figsize=(6,3)); x=np.arange(2); plt.bar(x-.15,acoustic,width=.3,label='acoustic'); plt.bar(x+.15,combined,width=.3,label='with LM'); plt.xticks(x,['cap','cat']); plt.legend(); plt.title('prior can flip the winner')
assert acoustic.argmax()==0 and combined.argmax()==1

In [ ]:
# 5) tiny spectrogram-to-symbol pipeline on synthetic vowel bands
sr=8000; t=np.arange(0,.12,1/sr); sig=np.r_[np.sin(2*np.pi*300*t[:320]),np.sin(2*np.pi*600*t[:320]),np.sin(2*np.pi*900*t[:320])]; S=stft_mag(sig,128,64); bands=[S[4:7].mean(0),S[9:12].mean(0),S[14:16].mean(0)]; pred=np.argmax(np.vstack(bands),axis=0)
plt.figure(figsize=(6,3)); plt.imshow(np.vstack(bands),aspect='auto',origin='lower'); plt.plot(pred,c='w'); plt.xlabel('frame'); plt.ylabel('symbol band'); plt.title('acoustic bands decode left-to-right')
assert pred[0]==0 and pred[-1]==2